## Линейная регрессия
В этом ноутбуке мы используем линейную регрессию для решения задачи предсказания цен на жильё в Бостоне.

Линейные методы предполагают, что между признаками объекта и целевой переменной существует линейная зависимость, то есть:
$$ y = w_1 x_1 + w_2 x_2 + ... + w_k x_k + b,$$
где у - целевая переменная (что мы хотим предсказать), $x_i$ -- признак объекта х, $w_i$ -- вес i-го признака, b -- bias (смещение, свободный член)

Часто предполагают, что объект х содержит в себе фиктивный признак, который всегда равен 1, тогда параметр $b$ соответствует весу этого признака в итоговой модели. В этом случае формула принимает простой вид:
$$ y = \langle w, x\rangle. $$

В матричной форме, в случае, когда у нас есть n объектов формулу можно переписать следующим образом:
$$ Y = Xw,$$
где Y --- вектор размера n, X --- матрица объекты-признаки размера $n \times k$, a w --- вектор весов размера k.

Решение по методу наименьших квадратов дает 
$$ w = (X^TX)^{-1}X^TY $$

### Загрузка датасета
Для начала загрузим набор данных, с которым мы будем работать. В библиотеке scikit-learn есть множество тренировочных наборов данных для освоения и проверки методов машинного обучения. Мы будем работать с датасетом Boston. Этот датасет описывает средние цены на недвижимость в микрорайонах Бостона в $1000. 
Примеры признаков микрорайона: количество преступлений на душу населения, процент старых домов в районе, среднее количество учеников на одного учителя и т.д.

Стандартные наборы данных в scikit-learn находятся в модуле sklearn.datasets.

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#dataset source: https://lib.stat.cmu.edu/datasets/boston

<YOUR CODE> описание датасета и ключевых признаков

### Выделение данных и анализ

Выделим матрицу объекты-признаки в переменную $X$, правильные ответы --- в переменную $y$. Используем библиотеку pandas. Для отображения информации о наборе данных используем функцию pd.describe, которая отображает полезные статистики из набора: средние значения признаков, минимум, максимум, медиану и др.

In [ ]:
<YOUR CODE>

Визуализируем распределения признаков в датасете

In [ ]:
<YOUR CODE>

Линейные алгоритмы часто сталкиваются с проблемой мультиколлинеарности признаков --- когда два или больше признаков оказываются близкими к линейно зависимым. Это, в частности, проявляется в том, что матрица $X$ становится необратимой, а, значит, в решении задачи линейной регрессией в формуле мы не можем вычислить $X^{-1}$. Эту проблему на практике обходят численными методами оптимизации, но решение задачи методом линейной регрессии по-прежнему может оказаться нестабильным. 

Проблему мультиколлинеарности устранить не так просто, но можно по крайней мере проверить, что никакие два признака не являются линейно зависимыми. В случае, если два признака оказываются линейно зависимыми, один из них (тот, который имеет меньшее влияние на целевую переменную $y$ может быть удалён из рассмотрения.

На практике линейную зависимость признаков можно выявить, посчитав матрицу корреляций признаков между собой. Сделать это нам поможет функция .corr() в padnas, а также модуль seaborn.

In [ ]:
import seaborn as sns

plt.figure(figsize=(10,7))
sns.heatmap(X.corr())

Если два признака имеют слишком большую или слишком маленькую корреляцию, их целесообразно удалить из датасета. Здесь лишь два признака имеют высокую корреляцию --- около 0.8. Один из этих признаков можно было бы удалить из рассмотрения, но мы сейчас это делать не будем, так как 0.8 --- не критичное значение.

Разобьём данные на train и test в соотношении 70:30. Используем функцию sklearn.model_selection.train_test_split.

In [ ]:
from sklearn.model_selection import train_test_split

<YOUR CODE>

Линейные алгоритмы восприимчивы к масштабу данных: они работают лучше, если все признаки имеют примерно одинаковую дисперсию. Отнормируем данные с помощью объекта StandardScaler из модуля sklearn.preprocessing.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

<YOUR CODE>

### Обучение линейной регрессии

Теперь всё готово для обучения. Линейная регрессия находится в модуле sklearn.linear_model. Обучим линейную регрессию на X_train и предскажем значения на X_test. 

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

<YOUR CODE>

Виуализируем получившиеся веса алгоритма.

In [ ]:
plt.figure(figsize=(20, 8))
plt.bar(X.columns, model.coef_)

### Оценка качества алгоритма

Для оценки качества работы алгоритма нам необходимы метрики. Мы посчитаем среднюю квадратичную ошиюбку: $$MSE = \frac{1}{n}\sum{(y_{true} - y_{pred})^2}$$ и среднюю абсолютную ошибку: $$MAE = \frac{1}{n}\sum{|y_{true} - y_{pred}|}.$$

Функции вычисления ошибок находятся в модуле sklearn.metrics.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

<YOUR CODE> для вывода Train MSE, Train MAE, Test MSE, Test MAE

In [ ]:
y.mean()

### Кросс-валидация
Иногда просто разбиение на обучающую и тестовую выборки не даёт точного прогноза оценки ошибки, ведь обученный алгоритм может сильно меняться в зависимости от обучающей выборки. Чтобы нивелировать эффект конкретной обучающей выборки, используют так называему кросс-валидацию. Идея кросс-валидации состоит в том, чтобы разбить все данные на несколько одинаковых по размеру частей, поочерёдно используя каждую часть как test, а оставшийся датасет --- как train. На каждом из экспериментов вычисляют тестовую ошибку, затем результат усредняют по всем экспериментам.

![alt text](https://drive.google.com/uc?id=1cS2AoSrcG5sCGbnKhkv3ZYqKSmtPd-XM)

Выполним эту схему на нашем датасете. Кросс-валидация находится в модуле sklearn.model_selection.

In [ ]:
from sklearn.model_selection import cross_val_score

result = cross_val_score(estimator=LinearRegression(), X=X, y=y, scoring='neg_mean_absolute_error', cv=5)
result

In [ ]:
print(f'Среднее MAE равно {-result.mean()}, стандартное отклонение MAE равно {result.std()}')